In [1]:
import sys

# Install PyMuPDF for PDF processing
!{sys.executable} -m pip install PyMuPDF

# Install pytesseract for OCR
!{sys.executable} -m pip install pytesseract Pillow

# Install Tesseract OCR engine
!sudo apt update
!sudo apt install tesseract-ocr

print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 52.5 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,914 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,618 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,011 kB]
Get:13 http://s

#Extract text from the pdf

In [2]:
import fitz  # PyMuPDF
from PIL import Image
import pytesseract
import os

pdf_path = '/content/History_of_Indian_painting.pdf'
output_text_file = 'extracted_text.txt'

# Check if the PDF file exists
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file '{pdf_path}' was not found.")
else:
    print(f"Processing PDF: {pdf_path}")
    text_data = []

    # Open the PDF
    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc):
            print(f"Extracting text from page {page_num + 1}/{len(doc)}")
            # Render page to an image (high resolution for better OCR)
            pix = page.get_pixmap(matrix=fitz.Matrix(300 / 72, 300 / 72))

            # Convert to PIL Image
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

            # Perform OCR
            page_text = pytesseract.image_to_string(img)
            text_data.append(f"--- Page {page_num + 1} ---\n{page_text}")

    # Save extracted text to a file
    with open(output_text_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(text_data))

    print(f"OCR complete. Extracted text saved to '{output_text_file}'.")

    # Display the first few lines of the extracted text
    with open(output_text_file, 'r', encoding='utf-8') as f:
        first_lines = [next(f) for _ in range(20)] # Read first 20 lines
    print("\n--- First 20 lines of extracted text ---")
    print(''.join(first_lines))
    print("-----------------------------------------")

Processing PDF: /content/History_of_Indian_painting.pdf
Extracting text from page 1/176
Extracting text from page 2/176
Extracting text from page 3/176
Extracting text from page 4/176
Extracting text from page 5/176
Extracting text from page 6/176
Extracting text from page 7/176
Extracting text from page 8/176
Extracting text from page 9/176
Extracting text from page 10/176
Extracting text from page 11/176
Extracting text from page 12/176
Extracting text from page 13/176
Extracting text from page 14/176
Extracting text from page 15/176
Extracting text from page 16/176
Extracting text from page 17/176
Extracting text from page 18/176
Extracting text from page 19/176
Extracting text from page 20/176
Extracting text from page 21/176
Extracting text from page 22/176
Extracting text from page 23/176
Extracting text from page 24/176
Extracting text from page 25/176
Extracting text from page 26/176
Extracting text from page 27/176
Extracting text from page 28/176
Extracting text from page 29/

#extract images from the pdf

In [5]:
import sys
!{sys.executable} -m pip install opencv-python
print("OpenCV installed.")

OpenCV installed.


In [6]:
import cv2
import numpy as np
import os
from PIL import Image

input_images_directory = 'extracted_images_illustrations' # Directory where the full page images are saved
output_illustrations_directory = 'refined_illustrations' # New directory for smaller illustrations

if not os.path.exists(output_illustrations_directory):
    os.makedirs(output_illustrations_directory)
    print(f"Created directory: {output_illustrations_directory}")

illustration_count = 0

print(f"Processing images from: {input_images_directory}")

for filename in os.listdir(input_images_directory):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
        image_path = os.path.join(input_images_directory, filename)

        # Load the image using OpenCV
        img = cv2.imread(image_path)
        if img is None:
            print(f"Could not load image: {image_path}")
            continue

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Apply Gaussian blur to reduce noise and help with contour detection
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)

        # Use adaptive thresholding to get a binary image
        # This is often good for scanned documents with varying lighting
        thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 11, 2)

        # Find contours
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Iterate through contours and filter potential illustrations
        img_height, img_width = img.shape[:2]
        min_illustration_area = (img_width * img_height) * 0.005 # Min area 0.5% of page
        max_illustration_area = (img_width * img_height) * 0.80 # Max area 80% of page

        for i, contour in enumerate(contours):
            x, y, w, h = cv2.boundingRect(contour)
            area = cv2.contourArea(contour)
            aspect_ratio = w / float(h) if h > 0 else 0

            # Filter based on area, aspect ratio, and size relative to page
            # You may need to tune these parameters for best results
            if (area > min_illustration_area and area < max_illustration_area and
                w > 50 and h > 50 and # Minimum absolute dimensions
                aspect_ratio > 0.1 and aspect_ratio < 10): # Reasonable aspect ratio

                # Crop the illustration region
                illustration = img[y:y+h, x:x+w]

                # Save the cropped illustration
                output_filename = os.path.join(output_illustrations_directory, f"{os.path.splitext(filename)[0]}_illustration_{i}.jpg")
                cv2.imwrite(output_filename, illustration)
                illustration_count += 1
                # print(f"  Extracted illustration {i} from {filename} as {output_filename}")

print(f"\nRefined image extraction complete. Total {illustration_count} potential illustrations extracted and saved to '{output_illustrations_directory}'.")

# List some of the extracted files to confirm
print("\n--- First 10 refined illustration files ---")
if os.path.exists(output_illustrations_directory):
    for i, filename in enumerate(os.listdir(output_illustrations_directory)):
        if i >= 10: break
        print(filename)
print("--------------------------------------")

Created directory: refined_illustrations
Processing images from: extracted_images_illustrations

Refined image extraction complete. Total 50 potential illustrations extracted and saved to 'refined_illustrations'.

--- First 10 refined illustration files ---
page_170_img_0_illustration_707.jpg
page_1_img_0_illustration_2827.jpg
page_97_img_0_illustration_832.jpg
page_132_img_0_illustration_368.jpg
page_176_img_0_illustration_1715.jpg
page_139_img_0_illustration_348.jpg
page_107_img_0_illustration_1163.jpg
page_61_img_0_illustration_723.jpg
page_86_img_0_illustration_396.jpg
page_51_img_0_illustration_623.jpg
--------------------------------------


**Download the extracted images folder as a .zip file**

In [7]:
import zipfile
import os
from google.colab import files

output_illustrations_directory = 'refined_illustrations'
zip_filename = 'refined_illustrations.zip'

if os.path.exists(output_illustrations_directory):
    print(f"Zipping folder '{output_illustrations_directory}'...")
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_in_dir in os.walk(output_illustrations_directory):
            for file in files_in_dir:
                file_path = os.path.join(root, file)
                # Add file to zip, preserving directory structure within the zip
                zipf.write(file_path, os.path.relpath(file_path, output_illustrations_directory))
    print(f"'{zip_filename}' created successfully.")

    # Offer to download the zip file
    files.download(zip_filename)
    print(f"'{zip_filename}' offered for download.")
else:
    print(f"Error: Directory '{output_illustrations_directory}' not found.")

Zipping folder 'refined_illustrations'...
'refined_illustrations.zip' created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'refined_illustrations.zip' offered for download.
